# Patrón Estructural: Decorator

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Decorator** permite **agregar responsabilidades** a un objeto de forma
**dinámica**, envolviéndolo en objetos que comparten su **misma interfaz**. Es una
alternativa flexible a la herencia para combinar funcionalidades.

### ¿Qué problema resuelve en la banca?
Una **cuenta** puede contratar servicios adicionales que suman una **cuota mensual**:
seguro de vida, tarjeta de crédito, alertas SMS... Las combinaciones son muchas
(seguro+SMS, tarjeta+seguro+SMS, etc.). Crear una subclase por cada combinación es
inmanejable. Decorator permite **apilar** servicios sobre la cuenta base.

## Código *sin patrón* (el problema es evidente)
Intentamos cubrir las combinaciones con subclases o con flags: explota combinatoriamente.

In [1]:
# Enfoque por subclases: una clase por combinacion -> explosion combinatoria
class CuentaBase:
    def descripcion(self): return "Cuenta base"
    def cuota_mensual(self): return 0

class CuentaConSeguro(CuentaBase):
    def descripcion(self): return "Cuenta base + Seguro"
    def cuota_mensual(self): return 15000

class CuentaConSMS(CuentaBase):
    def descripcion(self): return "Cuenta base + SMS"
    def cuota_mensual(self): return 5000

class CuentaConSeguroYSMS(CuentaBase):
    def descripcion(self): return "Cuenta base + Seguro + SMS"
    def cuota_mensual(self): return 20000

# ...y faltarian: Seguro+Tarjeta, SMS+Tarjeta, Seguro+SMS+Tarjeta, etc.

for c in [CuentaBase(), CuentaConSeguro(), CuentaConSMS(), CuentaConSeguroYSMS()]:
    print(f"{c.descripcion():35} -> cuota ${c.cuota_mensual()}")
print(">> Problema: N servicios producen 2^N combinaciones -> imposible mantener por herencia.")

Cuenta base                         -> cuota $0
Cuenta base + Seguro                -> cuota $15000
Cuenta base + SMS                   -> cuota $5000
Cuenta base + Seguro + SMS          -> cuota $20000
>> Problema: N servicios producen 2^N combinaciones -> imposible mantener por herencia.


### Análisis del problema
- Cada nueva combinación de servicios exige **otra subclase**.
- Con N servicios hay **2^N** combinaciones posibles: inmanejable.
- No se pueden **agregar/quitar** servicios en tiempo de ejecución.

## Código *con patrón* (problema resuelto)
Definimos una interfaz `Cuenta` y decoradores que **envuelven** una cuenta y suman su
propio costo/descripción. Los servicios se **apilan** libremente en tiempo de ejecución.

In [2]:
from abc import ABC, abstractmethod


# Componente
class Cuenta(ABC):
    @abstractmethod
    def descripcion(self) -> str: ...
    @abstractmethod
    def cuota_mensual(self) -> float: ...


# Componente concreto
class CuentaBasica(Cuenta):
    def descripcion(self) -> str: return "Cuenta base"
    def cuota_mensual(self) -> float: return 0


# Decorador base: mantiene la MISMA interfaz y envuelve una cuenta
class ServicioDecorator(Cuenta):
    def __init__(self, cuenta: Cuenta):
        self._cuenta = cuenta
    def descripcion(self) -> str: return self._cuenta.descripcion()
    def cuota_mensual(self) -> float: return self._cuenta.cuota_mensual()


# Decoradores concretos
class Seguro(ServicioDecorator):
    def descripcion(self) -> str: return self._cuenta.descripcion() + " + Seguro"
    def cuota_mensual(self) -> float: return self._cuenta.cuota_mensual() + 15000

class TarjetaCredito(ServicioDecorator):
    def descripcion(self) -> str: return self._cuenta.descripcion() + " + Tarjeta"
    def cuota_mensual(self) -> float: return self._cuenta.cuota_mensual() + 25000

class AlertasSMS(ServicioDecorator):
    def descripcion(self) -> str: return self._cuenta.descripcion() + " + SMS"
    def cuota_mensual(self) -> float: return self._cuenta.cuota_mensual() + 5000


# Se apilan libremente, en tiempo de ejecucion
cuenta = CuentaBasica()
cuenta = Seguro(cuenta)
cuenta = AlertasSMS(cuenta)
cuenta = TarjetaCredito(cuenta)

print(cuenta.descripcion())
print("Cuota mensual total: $", cuenta.cuota_mensual())

# Otra combinacion distinta sin crear ninguna clase nueva
otra = AlertasSMS(CuentaBasica())
print(otra.descripcion(), "-> $", otra.cuota_mensual())
print(">> Solucion: cualquier combinacion se arma apilando decoradores, sin nuevas clases.")

Cuenta base + Seguro + SMS + Tarjeta
Cuota mensual total: $ 45000
Cuenta base + SMS -> $ 5000
>> Solucion: cualquier combinacion se arma apilando decoradores, sin nuevas clases.


### Verificación
- Todos los decoradores comparten la interfaz `Cuenta` (`descripcion`, `cuota_mensual`).
- Se combinan **apilando** (`Seguro(AlertasSMS(CuentaBasica()))`), sin clases por combinación.
- Se pueden armar combinaciones nuevas en tiempo de ejecución sin tocar el código.

## UML del patrón Decorator
```plantuml
@startuml
interface Cuenta {
    + descripcion() : str
    + cuota_mensual() : float
}
class CuentaBasica
class ServicioDecorator {
    - _cuenta : Cuenta
}
Cuenta <|.. CuentaBasica
Cuenta <|.. ServicioDecorator
ServicioDecorator <|-- Seguro
ServicioDecorator <|-- TarjetaCredito
ServicioDecorator <|-- AlertasSMS
ServicioDecorator o--> Cuenta : envuelve
@enduml
```

## ¿Por qué Decorator y no otro patrón?
- El problema es **añadir responsabilidades combinables** (costo + descripción) a una
  cuenta sin caer en la explosión de subclases. Ese es el caso clásico de Decorator.
- No es Adapter (no cambiamos la interfaz, la **conservamos**) ni Facade (no simplificamos
  un subsistema). Aquí **enriquecemos** el objeto manteniendo su contrato.
- Frente a la herencia, Decorator permite **combinar** servicios en tiempo de ejecución y
  agregar un servicio nuevo con **una sola clase**, no con 2^N.